In [2]:
import numpy as np
import torch
import numpy.linalg as la
np.set_printoptions(formatter={"float":'{: 10.5f}'.format},linewidth=200)
log_path = 'log'
ref_path = 'npz_logging'
def concat(input_list, name, axis=2):
    return np.concatenate([x[name] for x in input_list], axis=axis)

c:\Users\ACCESS_user\anaconda3\envs\tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
idx = [[0,2],[1,1],[2,3]]
ord = idx[2]
add_op = np.load(log_path + "/" + str(ord[0]) + ".npz")
conv_op = np.load(ref_path + "/" + str(ord[1]) + "_conv.npz")
# add_ref_op = np.load(ref_path + "/" + str(ord[1] + 3) + "_add.npz")
# add_ref_next_op = np.load(ref_path + "/" + str(ord[1] + 5) + "_conv.npz")
print(conv_op["input_scale"])
#conv_op["input_scale"] = 1 / 128.0
print(conv_op["input_scale"])

x = add_op["in"][:,:1,...]
x_ref = conv_op["input"] / conv_op["input_scale"]
print(x)
print(x_ref)
print(x.shape)
print(x_ref.shape)
print ("input difference:", la.norm(x - x_ref[0,:,:,:]))
w = add_op["w"][:,:1,...]
w_ref = conv_op["w"] / conv_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))
b= add_op["b"]
b_ref = conv_op["b"] / conv_op["weight_scales"].flatten() / (1/128.0)#conv_op["input_scale"]
print ("b shape:", b.shape, "difference", la.norm(b - b_ref))
mac_res = add_op["mac_res"] + b.reshape(1,-1,1,1)
mac_res_ref = conv_op["output"][:1,:,:,:] / conv_op["weight_scales"].reshape(1,-1,1,1) / (1/128.0)#conv_op["input_scale"]
print ("mac_res shape:", mac_res.shape, "difference", la.norm(mac_res - mac_res_ref))
sIn, sMid, sOut, sAdd = add_op["scales"]
print ("sIn, sMid, sOut, sAdd", sIn, sMid, sOut, sAdd)
"""
print ("Ref: sIn, sMid, sOut, sAdd", 1/ conv_op["input_scale"], 1/ add_ref_op["a_scale"], sOut, sAdd)
mid_res = add_op["mid_res"]
mid_res_ref = add_ref_op["input1"] / add_ref_op["a_scale"]
print ("add_input1 difference:", la.norm(mid_res - mid_res_ref[0,:,:,:]))
elt_end = add_op["elt_end"] * sMid / sAdd
elt_end_ref = add_ref_op["input2"] / add_ref_op["input2_scale"]
print ("add_input2 difference:", la.norm(elt_end - elt_end_ref[0,:,:,:]))
out = add_op["out"]
out_ref = add_ref_next_op["input"] / add_ref_next_op["input_scale"]
print ("add_output difference:", la.norm(out - out_ref[0,:,:,:]), la.norm(out), la.norm(out_ref[0,:,:,:]))"
"""
import torch
import torch.nn as nn

conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=2, dilation=2, bias=True)
conv.bias = nn.Parameter(torch.from_numpy(b_ref).float())
conv.weight = nn.Parameter(torch.from_numpy(w_ref).float())
algo_mid_res = conv(torch.from_numpy(x).float())
print ("mac_res shape:", algo_mid_res.shape, "difference", la.norm(mac_res - algo_mid_res.detach().numpy()))
print(algo_mid_res)
print(add_op["mac_res"])

[   0.00781]
[   0.00781]
[[[[ -66 -122   77 ...  -88   -8   55]
   [-100   73  -61 ...  116  -61  108]
   [ 106  -55  -72 ...  117  120 -113]
   ...
   [ -27   99   35 ...   49  -45   23]
   [ -74 -116   51 ...    7  -26  104]
   [ -86  -77  -46 ...   67  105  116]]]]
[[[[ -66.00000 -122.00000   77.00000 ...  -88.00000   -8.00000   55.00000]
   [-100.00000   73.00000  -61.00000 ...  116.00000  -61.00000  108.00000]
   [ 106.00000  -55.00000  -72.00000 ...  117.00000  120.00000 -113.00000]
   ...
   [ -27.00000   99.00000   35.00000 ...   49.00000  -45.00000   23.00000]
   [ -74.00000 -116.00000   51.00000 ...    7.00000  -26.00000  104.00000]
   [ -86.00000  -77.00000  -46.00000 ...   67.00000  105.00000  116.00000]]]]
(1, 1, 64, 64)
(1, 1, 64, 64)
input difference: 0.0
w (cmodel) shape: (32, 1, 3, 3)
w shape: (32, 1, 3, 3) difference 0.0
b shape: (32,) difference 0.0
mac_res shape: (1, 32, 64, 64) difference 0.0
sIn, sMid, sOut, sAdd 128.0 2048.0 32.0 32.0
mac_res shape: torch.Size([

In [3]:
cmodel_op = np.load(log_path + "/1.npz")
torch_op = np.load(ref_path + "/5_add.npz")

x = cmodel_op["out"]
x_ref = torch_op["a"] / torch_op["a_scale"]
print(x.shape)
print(x_ref.shape)

print ("x difference", la.norm(x - x_ref))

(1, 32, 64, 64)
(1, 32, 64, 64)
x difference 0.0


In [3]:
cmodel_op = np.load(log_path + "/2.npz")
torch_op = np.load(ref_path + "/6_conv.npz")

x = cmodel_op["out"]
x_ref = torch_op["input"] / torch_op["input_scale"]
print ("x difference", la.norm(x - x_ref))
print(x)
print(x_ref)

x difference 0.0
[[[[ -98  -34   18 ...  -61 -105  -25]
   [   4   37  -48 ...   85  -52   55]
   [ 123  -63   -4 ...  127   56   12]
   ...
   [ -24  123   -1 ...  -29 -107 -128]
   [ -73  -88   22 ... -111  -36   29]
   [ -40  -57  -54 ...   30  127   95]]

  [[ -60  -14  -74 ...   68 -128  -25]
   [ -48  -37   85 ...   89   13  -66]
   [ -80  -82    1 ...   17    6   87]
   ...
   [ -48  -50   24 ...   43  -39 -110]
   [  12  -22   -8 ...  -32 -119   18]
   [ -60   -9   12 ...   45   69   11]]

  [[ 117  127  -39 ...  127  127   -1]
   [ 127   39  127 ...   52  127  -45]
   [ -84  127  127 ...  106 -101  127]
   ...
   [  86  -76  127 ...   72  127   30]
   [ 120  127  -12 ...  127  127   39]
   [ 127  127  127 ...   -6   58    9]]

  ...

  [[ -20  127   52 ...  -36  122   -4]
   [  44  119  -24 ... -128   98   53]
   [  46 -128   79 ...   55 -128  127]
   ...
   [ -24 -128 -128 ...  127   50  -34]
   [  86 -128  112 ...  -62   74    5]
   [ 124  116  127 ... -128   64  -50]]

  [[

In [5]:
cmodel_op = np.load(log_path + "/4.npz")
torch_op = np.load(ref_path + "/8_conv.npz")

print(cmodel_op.files)
x = cmodel_op["in"]
x_ref = torch_op["input"] / torch_op["input_scale"]
print ("x difference", la.norm(x.reshape(x_ref.shape) - x_ref))
print(x)
print(x_ref)

['stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w', 'stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w']
x difference 0.0
[[[[ 96 127  15 ...   0 127 115]
   [-11 -10  27 ...   0   0  52]
   [-11  -5 118 ...  43   0   0]
   ...
   [ 97  76   0 ...  62 127  85]
   [-11 127  44 ...   0 -11 -10]
   [ -1 108  24 ...  -2   0   0]]

  [[ 76   0  36 ... -11  -5   2]
   [ -3 115  53 ...  25  -9  -9]
   [  0   0   2 ... -10  -9   0]
   ...
   [  0  43   0 ... 106 127  -5]
   [ -9 -11  -6 ... 110  -7  -2]
   [ 22 127  -6 ... -11  -3  -5]]

  [[  0 127  27 ...   0 -12 127]
   [ 46 -11 -11 ...   0   0  64]
   [127 127  27 ...  27  -2 -11]
   ...
   [127 127   9 ...  12 -11  -2]
   [ 49 127 127 ...   0 -10 -11]
   [ -7   0  -4 ...  -1 -10  78]]

  ...

  [[-11   0   6 ... -12   0  -1]
   [ -7 -11  -7 ...   6 -11  28]
   [ 81  34   0 ...  36  15   0]
   ...
   [  6   7  -4 ...  -8  -2  -

In [7]:
cmodel_op = np.load(log_path + "/13.npz")
torch_op = np.load(ref_path + "/40_conv.npz")

print(cmodel_op.files)
x = cmodel_op["in"]
print(torch_op.files)
x_ref = torch_op["input"] / torch_op["input_scale"]
print ("x difference", la.norm(x - x_ref))
print(x)
print(x_ref)

['stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w', 'stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w']
['w', 'b', 'input_scale', 'weight_scales', 'input', 'output']
x difference 0.0
[[[[ -3 -11  -5 ...  -7  -9 -11]
   [  3  -9  10 ...  -5 -10  -6]
   [  2  -9  -7 ...   7   7  -8]
   ...
   [  2  -5 -11 ...  -3  -9  -2]
   [  6 -11  -2 ...  -4 -11 -11]
   [ 31   5  -5 ...  40  32  45]]

  [[ 12  10   6 ...   3  -5  56]
   [  7  15   1 ...  25  18  -2]
   [ -4  17  32 ...  -3   7  40]
   ...
   [ 12  13  15 ...  16   6  20]
   [  8   6   5 ...   0  10  34]
   [ 12   7   3 ...   7  -5  -7]]

  [[  0 -11 -11 ... -11 -11  -7]
   [  3  -9 -11 ... -11  -4  -6]
   [ -3  -6  -9 ... -10  -7 -11]
   ...
   [ -2  -9 -11 ... -10 -11  -4]
   [ 10  -1  -8 ...  -7 -11 -10]
   [ -3  -7  -8 ...  13  12   4]]

  ...

  [[ 84  95 103 ...  98  75  42]
   [ 86 108  31 ...  51  27  60]
   [ 98 

In [8]:
cmodel_op = np.load(log_path + "/20.npz")
torch_op = np.load(ref_path + "/29_conv.npz")

print(cmodel_op.files)
x = cmodel_op["in"]
print(torch_op.files)
x_ref = torch_op["input"] / torch_op["input_scale"]
print ("x difference", la.norm(x - x_ref))
print(x)
print(x_ref)

w = cmodel_op["w"]
w_ref = torch_op["w"] / torch_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))

['stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w', 'stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w']
['w', 'b', 'input_scale', 'weight_scales', 'input', 'output']
x difference 0.0
[[[[-22 -22 -19 ... -18 -21  -7]
   [-12   0   0 ... -23  -7 -19]
   [-13  -3 -12 ... -16 -23 -21]
   ...
   [-13 -12 -23 ... -22 -17  -8]
   [ -5 -12 -22 ... -12   2  80]
   [-23 -22 -19 ...  22  75  74]]

  [[  4  60  49 ...  19  61  52]
   [ 44  91  68 ...  45  65  65]
   [ 46  95  72 ...  68  84  75]
   ...
   [ 38 123  63 ...   8  93  52]
   [ 30  83  64 ...  19  66  26]
   [ 32 120  57 ...  23   5  -6]]

  [[ 21  23  12 ...  18  28  16]
   [ 51  36  31 ...  34  26  31]
   [ 28  29  31 ...  62  35  -6]
   ...
   [ 35  25  23 ... -14  16   6]
   [ 42  80  37 ...  19  30  16]
   [ 46  72  55 ...   4  54  65]]

  ...

  [[127 115  92 ...  14   4  -1]
   [127 118 101 ...  23  12   6]
   [127 

In [9]:
add_op = np.load(log_path + "/18.npz")
conv_op = np.load(ref_path + "/49_conv.npz")

print(conv_op["input_scale"])

x = add_op["in"]
x_ref = conv_op["input"] / conv_op["input_scale"]
print(x.shape)
print(x_ref.shape)
print ("input difference:", la.norm(x - x_ref))
w = add_op["w"]
w_ref = conv_op["w"] / conv_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))
b= add_op["b"]
b_ref = conv_op["b"] / conv_op["weight_scales"].flatten() / conv_op["input_scale"]
print ("b shape:", b.shape, "difference", la.norm(b - b_ref))
mac_res = add_op["mac_res"] + b.reshape(1,-1,1,1)
mac_res_ref = conv_op["output"][:1,:,:,:] / conv_op["weight_scales"].reshape(1,-1,1,1) / conv_op["input_scale"]
print ("mac_res shape:", mac_res.shape, "difference", la.norm(mac_res - mac_res_ref))
sIn, sMid, sOut, sAdd = add_op["scales"]
print ("sIn, sMid, sOut, sAdd", sIn, sMid, sOut, sAdd)

[   0.03125]
(1, 64, 32, 32)
(1, 64, 32, 32)
input difference: 0.0
w (cmodel) shape: (32, 64, 1, 1)
w shape: (32, 64, 1, 1) difference 0.0
b shape: (32,) difference 0.0
mac_res shape: (1, 32, 32, 32) difference 0.0
sIn, sMid, sOut, sAdd 32.0 1024.0 1024.0 0.0


In [7]:
cmodel_op = np.load(log_path + "/38.npz")
torch_op = np.load(ref_path + "/65_conv.npz")

print(cmodel_op.files)
x = cmodel_op["in"]
print(torch_op.files)
x_ref = torch_op["input"] / torch_op["input_scale"]
print ("x difference", la.norm(x[:,:32,...] - x_ref[:,:32,...]))
print(x[:,32:,...])
print(x_ref[:,32:,...])

print(np.max(x[:,:32,...] - x_ref[:,:32,...]))

w = cmodel_op["w"]
w_ref = torch_op["w"] / torch_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))

print()

['stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w', 'stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w']
['w', 'b', 'input_scale', 'weight_scales', 'input', 'output']
x difference 2.6457513110645907
[[[[  3   8  16 ...   8   0  -3]
   [ 10  16  28 ...  30  13   4]
   [ 23  32  50 ...  74  37  19]
   ...
   [ 10  14  23 ...  21  31  36]
   [ 11  15  23 ...  22  25  27]
   [ 11  15  23 ...  22  22  22]]

  [[-17 -18 -22 ... -22 -22 -21]
   [-18 -19 -21 ... -23 -22 -22]
   [-18 -19 -21 ... -23 -23 -22]
   ...
   [-22 -22 -22 ... -21 -21 -22]
   [-21 -21 -22 ... -18 -20 -21]
   [-20 -21 -22 ... -17 -20 -21]]

  [[ 22  27  37 ...  30  22  18]
   [ 23  30  44 ...  30  22  18]
   [ 25  36  57 ...  31  22  18]
   ...
   [ 22  26  34 ...  33  24  20]
   [ 20  22  26 ...  31  27  24]
   [ 18  19  22 ...  30  28  27]]

  ...

  [[ 23  26  30 ...  28  22  20]
   [ 24  26  32 ...  29  2

In [8]:
cmodel_op = np.load(log_path + "/19.npz")
ref = np.load(ref_path + "/51_resize.npz")

x = cmodel_op["out"]
ref_x = ref["x_r_q_out"] / ref["input_scale"]
#print(x)
#print(ref_x.shape)
print ("x difference", la.norm(x - ref_x))

x difference 0.0


In [9]:
cmodel_op = np.load(log_path + "/9.npz")
ref = np.load(ref_path + "/24_resize.npz")

x = cmodel_op["out"]
ref_x = ref["x_r_q_out"] / ref["input_scale"]
#print(x)
#print(ref_x.shape)
print ("x difference", la.norm(x - ref_x))

x difference 0.0


In [10]:
add_op = np.load(log_path + "/24.npz")
conv_op = np.load(ref_path + "/35_conv.npz")

print(conv_op["input_scale"])

x = add_op["in"]
x_ref = conv_op["input"] / conv_op["input_scale"]
print(x.shape)
print(x_ref.shape)
print ("input difference:", la.norm(x - x_ref))
w = add_op["w"]
w_ref = conv_op["w"] / conv_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))
b= add_op["b"]
b_ref = conv_op["b"] / conv_op["weight_scales"].flatten() / conv_op["input_scale"]
print ("b shape:", b.shape, "difference", la.norm(b - b_ref))
mac_res = add_op["mac_res"] + b.reshape(1,-1,1,1)
mac_res_ref = conv_op["output"][:1,:,:,:] / conv_op["weight_scales"].reshape(1,-1,1,1) / conv_op["input_scale"]
print ("mac_res shape:", mac_res.shape, "difference", la.norm(mac_res - mac_res_ref))
sIn, sMid, sOut, sAdd = add_op["scales"]
print ("sIn, sMid, sOut, sAdd", sIn, sMid, sOut, sAdd)

[   0.01562]
(1, 32, 64, 64)
(1, 32, 64, 64)
input difference: 1.0
w (cmodel) shape: (32, 32, 3, 3)
w shape: (32, 32, 3, 3) difference 0.0
b shape: (32,) difference 0.0
mac_res shape: (1, 32, 64, 64) difference 945.8111366318108
sIn, sMid, sOut, sAdd 64.0 1024.0 1024.0 0.0


In [7]:
from quantizer.lsq import batch_frexp
import torch.nn.functional as F

add_op = np.load(log_path + "/23.npz")
conv_op = np.load(ref_path + "/33_conv.npz")

print(add_op.files)
print(conv_op["input_scale"])
print(add_op["dilation"])
print(add_op["stride"])
print(add_op["pad"])


sIn, sMid, sOut, sAdd = add_op["scales"]
w_scale = ((np.power(2, add_op['exponents'])/ add_op['mantissa'])) * 8.0#sOut / sIn

x = add_op["in"]
x_ref = conv_op["input"] / conv_op["input_scale"]
print(x.shape)
print(x_ref.shape)
print ("input difference:", la.norm(x - x_ref))
w = add_op["w"]
w_ref = conv_op["w"] / conv_op["weight_scales"]
print ("w (cmodel) shape:", w.shape)
print ("w shape:", w_ref.shape, "difference", la.norm(w - w_ref))
b= add_op["b"]
b_ref = conv_op["b"] / conv_op["weight_scales"].flatten() / conv_op["input_scale"]
print ("b shape:", b.shape, "difference", la.norm(b - b_ref))
mac_res = add_op["mac_res"] + b.reshape(1,-1,1,1)
mac_res_ref = conv_op["output"][:1,:,:,:] / (conv_op["weight_scales"]).reshape(1,-1,1,1) / conv_op["input_scale"]
print ("mac_res shape:", mac_res.shape, "difference", la.norm(mac_res - mac_res_ref.round()))
mid_res = add_op["mid_res"]
mid_res_ref = mac_res_ref / ((np.power(2, add_op['exponents']).reshape(1,-1,1,1) /add_op['mantissa'].reshape(1,-1,1,1)))
print(mid_res)
print(mid_res_ref)
print ("mid_res shape:", mid_res.shape, "difference", la.norm(mid_res - mid_res_ref.round()))


print ("sIn, sMid, sOut, sAdd", sIn, sMid, sOut, sAdd)

print(add_op['mantissa'])
print(add_op['exponents'])

m, e = batch_frexp(torch.from_numpy(conv_op["weight_scales"].flatten()))
print(m)
print(e - 3)

print(add_op['mantissa'] - m.numpy())
print(add_op['exponents'] - (e.numpy()-3))


w_scale_ref = 1/ conv_op["weight_scales"].astype('float64').flatten()
print(conv_op["weight_scales"].flatten())
print(w_scale)
print(w_scale_ref)
print ("w scale shape:", w_scale_ref.shape, "difference", la.norm(w_scale - w_scale_ref))

diff_mask = mac_res != mac_res_ref

# Get the values from both arrays where they differ
mac_res_values = mac_res[diff_mask]
mac_res_ref_values = mac_res_ref[diff_mask]
# Get the indices where they differ
diff_indices = np.where(diff_mask)

weight_integer = torch.from_numpy(w_ref)
bias_integer = torch.from_numpy(b_ref)
output2 = F.conv2d(torch.from_numpy(x).float(), weight_integer, bias_integer, 1, 1, 1, 1)
print ("output2 shape:", output2.shape, "difference", la.norm(mac_res - output2.numpy()))
output2 = output2[0,...][diff_mask]
# Print the differences
for idx, (val, ref, conv) in enumerate(zip(mac_res_values, mac_res_ref_values, output2)):
    print(f"Index {tuple(i[idx] for i in diff_indices)}: mac_res_values={val}, mac_res_ref_values={ref}, conv_value={conv}")
    if idx > 10:#(abs(p_val - c_val) == 3):
        break

# Optional: Print total number of differences
print(f"\nTotal differences: {np.sum(diff_mask)}")


#print(output2)

['stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w', 'stride', 'pad', 'dilation', 'scales', 'mac_res', 'mantissa', 'exponents', 'b', 'mid_res', 'out', 'in', 'w']
[   0.00781]
[1]
[1]
[1 1 1 1]
(1, 32, 64, 64)
(1, 32, 64, 64)
input difference: 0.0
w (cmodel) shape: (32, 32, 3, 3)
w shape: (32, 32, 3, 3) difference 0.0
b shape: (32,) difference 0.0
mac_res shape: (1, 32, 64, 64) difference 4.58257569495584
[[[[ -673  -848  -944 ...  -551  -516  -323]
   [-1116 -1638 -1604 ...  -976 -1261  -781]
   [-1227 -1612 -1583 ... -1168 -1073 -1006]
   ...
   [-1161 -1370 -1448 ... -1349 -1243  -879]
   [ -974 -1191 -1176 ... -1055  -738  -357]
   [ -914  -847  -903 ...  -419  -190  -429]]

  [[ -855 -1240 -1301 ... -1141  -968  -885]
   [-1460 -1885 -1827 ... -1504 -1407 -1055]
   [-1388 -1684 -1852 ... -1555 -1580 -1021]
   ...
   [-1308 -1576 -1582 ... -1677 -1431  -871]
   [-1321 -1471 -1567 ... -1176 -1098  -609]
   [-1025  -928  -835 ..

Compare Network Output

In [2]:
add_op = np.load(log_path + "/100.npz")
print(add_op["out"])
conv_op = np.load(ref_path + "/190_conv.npz")
print(conv_op["output"] * 16)
print ("x difference", la.norm(add_op["out"] - conv_op["output"]))
print ("x difference", la.norm(np.sign(add_op["out"]) - np.sign(conv_op["output"])))

print(np.sign(add_op["out"]))
print(np.sign(conv_op["output"]))

print(np.sum(np.sign(add_op["out"])))

[[[[-78 -80 -74 ... -54 -53 -53]
   [-82 -70 -60 ... -54 -55 -58]
   [-79 -62 -50 ... -49 -54 -59]
   ...
   [-69 -58 -50 ... -49 -57 -62]
   [-68 -58 -49 ... -50 -54 -62]
   [-68 -70 -71 ... -64 -66 -64]]

  [[  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   ...
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]]

  [[  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   ...
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]]

  ...

  [[  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   ...
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]]

  [[  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   0   0]
   ...
   [  0   0   0 ...   0   0   0]
   [  0   0   0 ...   0   